# Fine-tune a Piper voice from the consented speaker's clips (Google Colab, free T4 GPU)

1. *Runtime → Change runtime type → T4 GPU*.
2. Upload `voice_clone/work/dataset/<voice>.zip` to **Google Drive → My Drive/voice_clone/**.
3. Set `VOICE` and `LANG` in the next cell, then *Runtime → Run all*.
4. Training saves to Drive after every epoch. When Colab disconnects (free sessions last a few hours), *Run all*
   again — it resumes from the last checkpoint.
5. When the test sentence in cell 7 sounds like the speaker, run cell 8 and install the files with
   `voice_clone/install_voice.py`.

The voice starts from Piper's **LJSpeech** checkpoint (public-domain data) and learns the speaker from the clips.
Training code: piper1-gpl (GPL-3.0) — used here only to produce the model file.
**Not run on the project laptop (no NVIDIA GPU).** Commands follow piper1-gpl's TRAINING.md; if a cell fails, the
error usually names the missing package or option.

In [ ]:
VOICE = "hans_hi"          # name of the zip from make_dataset.py (without .zip)
LANG = "hi"                # espeak-ng voice: "hi" for Hindi/Hinglish, "en-us" for English
EXTRA_EPOCHS = 1500        # more = closer to the speaker (and longer); 1000-3000 is typical for 30-60 min of audio
DRIVE_DIR = "/content/drive/MyDrive/voice_clone"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, zipfile
os.makedirs(f"{DRIVE_DIR}/{VOICE}_run", exist_ok=True)
with zipfile.ZipFile(f"{DRIVE_DIR}/{VOICE}.zip") as z:
    z.extractall("/content/data")
print(open("/content/data/dataset_info.json").read())

In [ ]:
%%bash
set -e
apt-get -qq install -y build-essential cmake ninja-build > /dev/null
cd /content
[ -d piper1-gpl ] || git clone -q https://github.com/OHF-Voice/piper1-gpl
cd piper1-gpl
pip -q install -e ".[train]"
./build_monotonic_align.sh
python3 setup.py build_ext --inplace > /dev/null
echo installed

In [ ]:
import glob, torch
from huggingface_hub import hf_hub_download
runs = sorted(glob.glob(f"{DRIVE_DIR}/{VOICE}_run/**/last.ckpt", recursive=True) +
              glob.glob(f"{DRIVE_DIR}/{VOICE}_run/**/*.ckpt", recursive=True), key=os.path.getmtime)
if runs:
    CKPT = runs[-1]                                   # resume
else:
    CKPT = hf_hub_download("rhasspy/piper-checkpoints", "en/en_US/ljspeech/medium/lj-med_1000.ckpt")
base_epoch = torch.load(CKPT, map_location="cpu", weights_only=False).get("epoch", 0)
target_file = f"{DRIVE_DIR}/{VOICE}_run/max_epochs.txt"
if runs:                                    # resuming: keep the target set on the first run
    MAX_EPOCHS = int(open(target_file).read())
else:                                       # the base checkpoint already has its own epochs
    MAX_EPOCHS = base_epoch + EXTRA_EPOCHS
    open(target_file, "w").write(str(MAX_EPOCHS))
print("checkpoint:", CKPT, "| epoch", base_epoch, "| train until", MAX_EPOCHS)

In [ ]:
%cd /content/piper1-gpl
!python3 -m piper.train fit \
  --data.voice_name "{VOICE}" \
  --data.csv_path /content/data/metadata.csv \
  --data.audio_dir /content/data/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice "{LANG}" \
  --data.cache_dir /content/cache \
  --data.config_path /content/data/config.json \
  --data.batch_size 16 \
  --trainer.max_epochs {MAX_EPOCHS} \
  --trainer.default_root_dir "{DRIVE_DIR}/{VOICE}_run" \
  --ckpt_path "{CKPT}"

In [ ]:
import glob, os, shutil, subprocess
from IPython.display import Audio, display
last = sorted(glob.glob(f"{DRIVE_DIR}/{VOICE}_run/**/*.ckpt", recursive=True), key=os.path.getmtime)[-1]
!python3 -m piper.train.export_onnx --checkpoint "{last}" --output-file "/content/{VOICE}.onnx"
shutil.copy("/content/data/config.json", f"/content/{VOICE}.onnx.json")
!pip -q install piper-tts
text = "नमस्ते! Hans Hyundai में आपका स्वागत है।" if LANG == "hi" else "Namaste! Welcome to Hans Hyundai."
subprocess.run(["piper", "-m", f"/content/{VOICE}.onnx", "-f", "/content/test.wav"], input=text.encode(), check=True)
display(Audio("/content/test.wav"))
print("checkpoint:", last)

In [ ]:
from google.colab import files
files.download(f"/content/{VOICE}.onnx")
files.download(f"/content/{VOICE}.onnx.json")